<a href="https://colab.research.google.com/github/locbp-uzh/biopipelines/blob/main/ExamplePipelines/notebooks/ubiquitin_results.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Inverse Folding (Ubiquitin)

**BioPipelines example** — inverse folding of ubiquitin using ProteinMPNN. Designed sequences are refolded with AlphaFold2 and filtered by RMSD and pLDDT. Sequences passing the filter are codon-optimised for *E. coli* expression with DNAEncoder.

[![Documentation](https://img.shields.io/badge/docs-readthedocs-blue)](https://biopipelines.readthedocs.io/en/latest/)
[![Preprint](https://img.shields.io/badge/preprint-bioRxiv-B31B1B)](https://www.biorxiv.org/content/10.64898/2026.03.11.711024v1)

In [1]:
# Cell 1: Install BioPipelines and micromamba
!git clone https://github.com/locbp-uzh/biopipelines
%cd biopipelines
!pip install -e ".[all]"
!wget -q https://github.com/mamba-org/micromamba-releases/releases/latest/download/micromamba-linux-64 -O /usr/local/bin/micromamba && chmod +x /usr/local/bin/micromamba
!micromamba create -f Environments/biopipelines.yaml -y

Cloning into 'biopipelines'...
remote: Enumerating objects: 7291, done.
remote: Counting objects: 100% (190/190), done.
remote: Compressing objects: 100% (60/60), done.
remote: Total 7291 (delta 137), reused 130 (delta 130), pack-reused 7101 (from 1)
Receiving objects: 100% (7291/7291), 14.39 MiB | 28.34 MiB/s, done.
Resolving deltas: 100% (5736/5736), done.
/content/biopipelines
Obtaining file:///content/biopipelines
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 42.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 36.7/36.7 MB 65.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 87.4 MB/s eta 0:00:00
  Building editable for biopipelines (pyproject.toml) ... done
  Created wheel for biopipelines: filename=biopipelines-1.1.0-0.editable-p

In [2]:
# Cell 2: Install tools
from biopipelines.pipeline import *
from biopipelines.rfdiffusion import RFdiffusion
from biopipelines.protein_mpnn import ProteinMPNN
from biopipelines.alphafold import AlphaFold
from biopipelines.pymol import PyMOL

with Pipeline(project="Setup", job="InstallTools"):
    RFdiffusion.install(weights=[])  # SE3nv environment only, no weights (required by ProteinMPNN)
    ProteinMPNN.install()
    AlphaFold.install()
    PyMOL.install() # For structure-based analysis like ConformationalChange


Running RFdiffusion_installation (step 1)
=== Activating Environment ===
Requested: biopipelines
Environment: biopipelines
Location: /root/.local/share/mamba/envs/biopipelines
Python: /root/.local/share/mamba/envs/biopipelines/bin/python
Python version: Python 3.12.13
=== Installing RFdiffusion (micromamba) ===
Cloning into 'RFdiffusion'...
conda-forge/linux-64                                        Using cache
conda-forge/noarch                                          Using cache


Transaction

  Prefix: /root/.local/share/mamba/envs/SE3nv

  Updating specs:

   - python=3.9


  Package               Version  Build                 Channel           Size
───────────────────────────────────────────────────────────────────────────────
  Install:
───────────────────────────────────────────────────────────────────────────────

  + _openmp_mutex           4.5  20_gnu                conda-forge     Cached
  + bzip2                 1.0.8  hda65f42_9            conda-forge     Cached
  + ca-

## Cell 3: Inverse Folding Pipeline

Fetches ubiquitin (PDB: 4LCD, chain E) and generates 10 sequences with ProteinMPNN (soluble model).
AlphaFold2 refolds each sequence; those with RMSD < 1.5 Å and pLDDT > 80 are codon-optimised for *E. coli* expression.

In [3]:
# Cell 3: Pipeline
from biopipelines.pipeline import *
from biopipelines.protein_mpnn import ProteinMPNN
from biopipelines.alphafold import AlphaFold
from biopipelines.conformational_change import ConformationalChange
from biopipelines.panda import Panda
from biopipelines.dna_encoder import DNAEncoder

with Pipeline(project="Ubiquitin", job="InverseFolding"):
    ubiquitin = PDB("4LCD", chain="E")
    sequences = ProteinMPNN(structures=ubiquitin,
                            num_sequences=10,
                            soluble_model=True)
    folded = AlphaFold(proteins=sequences)
    dna = DNAEncoder(sequences=sequences,
                     organism="EC")
    conf_change = ConformationalChange(reference_structures=ubiquitin,
                                       target_structures=folded)
    filtered_sequences = Panda(tables=[folded.tables.confidence,
                                       conf_change.tables.changes],
                               operations=[Panda.merge(),
                                            Panda.filter("RMSD < 1.5 and plddt > 80")],
                               pool=sequences)
    dna = DNAEncoder(sequences=filtered_sequences,
                     organism="EC")
dna

  4LCD not found locally, will download from RCSB

Running PDB (step 1)
=== Activating Environment ===
Requested: biopipelines
Environment: biopipelines
Location: /root/.local/share/mamba/envs/biopipelines
Python: /root/.local/share/mamba/envs/biopipelines/bin/python
Python version: Python 3.12.13
Fetching 1 structures
Convert: keep as-is (pdb|cif)
PDB IDs: 4LCD
Custom IDs: 4LCD
Priority: PDBs/ -> RCSB download
Output folder: /content/biopipelines/tests/Ubiquitin/InverseFolding_001/001_PDB
Fetching 1 structures (keep as-is (pdb|cif))
Priority: PDBs/ -> RCSB download
Water molecules will be removed

[1/1] Processing 4LCD -> 4LCD
4LCD not found locally, downloading from RCSB
Cached to PDBs/ folder: /content/biopipelines/PDBs/4LCD.pdb
Successfully downloaded 4LCD as 4LCD: 779382 bytes (rcsb_download (PDB))

Successful fetches saved: /content/biopipelines/tests/Ubiquitin/InverseFolding_001/001_PDB/structures.csv (1 structures)
Sequences saved: /content/biopipelines/tests/Ubiquitin/InverseF

StandardizedOutput({'sequences': DataStream(name='sequences', format='dna', items=10, files=0, map_table=set), 'tables': {'dna': TableInfo(name='dna', path='/content/biopipelines/tests/Ubiquitin/InverseFolding_001/007_DNAEncoder/dna.csv', columns=['id', 'protein_sequence', 'dna_sequence', 'organism', 'method'], count=1)}, 'output_folder': '/content/biopipelines/tests/Ubiquitin/InverseFolding_001/007_DNAEncoder', 'excel': '/content/biopipelines/tests/Ubiquitin/InverseFolding_001/007_DNAEncoder/dna_sequences.xlsx', 'info': '/content/biopipelines/tests/Ubiquitin/InverseFolding_001/007_DNAEncoder/dna_info.txt'})